In [ ]:
"""Get the residual stream activations from a model.
From 'Towards Monosemanticity' (https://transformer-circuits.pub/2023/monosemantic-features/index.html#appendix-autoencoder-dataset):
To create the dataset for autoencoder training, we evaluate the transformers on 40 million contexts from the Pile and collect the MLP activation vectors after the ReLU for each token within each context. 
We then sample activation vectors from 250 tokens in each context and shuffle these together so that samples within a batch come from diverse contexts."""
import torch
from utils.activation_hooks import ResidualStreamCollector
from transformers import AutoModelForCausalLM, AutoTokenizer
checkpoint = "HuggingFaceTB/SmolLM2-135M-Instruct"

device = "cuda" 
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# for multiple GPUs install accelerate and do `model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto")`
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

middle_layer = model.config.num_hidden_layers // 2

print(middle_layer)

# Example usage:
def collect_activations(model, tokenizer, text: str, layer_idx: int) -> torch.Tensor:
    """Collect residual stream activations for given text."""
    collector = ResidualStreamCollector(model, layer_idx)
    collector.attach_hook()
    
    # Tokenize and move to model's device
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Forward pass with no grad
    with torch.no_grad():
        model(**inputs)
    
    # Get activations and cleanup
    activations = collector.get_activations()
    collector.remove_hook()
    
    return activations

# Test it:
if __name__ == "__main__":
    # Using your model setup from above
    text = "The Golden Gate Bridge is"
    activations = collect_activations(model, tokenizer, text, middle_layer)
    print(f"Collected activations shape: {activations[0].shape}")


15
Collected activations shape: torch.Size([1, 5, 576])
